# 04 - Advanced Modeling, Hyperparameter Tuning & Model Export
## Building Energy Efficiency: Heating & Cooling Load Prediction
**DEPI Graduation Project - Team CloudXAi (NHA-5-165)**

This notebook covers:
1. Training Gradient Boosting Regressor
2. Training Default XGBoost
3. Hyperparameter Tuning using `RandomizedSearchCV` on XGBoost
4. Feature Importance Analysis
5. Saving all 10 candidate models & creating the Production Model Bundle.


In [ ]:
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import RandomizedSearchCV, cross_val_score
from xgboost import XGBRegressor

from src.energy_efficiency.config import TARGETS, RANDOM_STATE, CANDIDATES_DIR, PRODUCTION_DIR
from src.energy_efficiency.data_loader import load_processed_data
from src.energy_efficiency.preprocessing import build_preprocessor
from src.energy_efficiency.evaluation import evaluate_model
from src.energy_efficiency.model_export import create_model_bundle

print("Advanced modeling environment initialized.")


### 1. Load Processed Data


In [ ]:
train_df, test_df = load_processed_data()
features_cols = [c for c in train_df.columns if c not in TARGETS]
X_train, y_train = train_df[features_cols], train_df[TARGETS]
X_test, y_test = test_df[features_cols], test_df[TARGETS]

prep_tree = build_preprocessor(scale=False, onehot=False, engineer=True).fit(X_train)
X_train_tr = prep_tree.transform(X_train)
X_test_tr = prep_tree.transform(X_test)


### 2. Gradient Boosting & Default XGBoost


In [ ]:
gb = MultiOutputRegressor(GradientBoostingRegressor(random_state=RANDOM_STATE))
gb.fit(X_train_tr, y_train)

xgb_default = MultiOutputRegressor(XGBRegressor(random_state=RANDOM_STATE, n_jobs=1, verbosity=0))
xgb_default.fit(X_train_tr, y_train)

adv_results = []
for name, model in [("Gradient Boosting", gb), ("XGBoost (Default)", xgb_default)]:
    preds = model.predict(X_test_tr)
    for i, tgt in enumerate(TARGETS):
        adv_results.append(evaluate_model(y_test[tgt], preds[:, i], tgt, name))

display(pd.DataFrame(adv_results))


### 3. Hyperparameter Tuning for XGBoost per Target


In [ ]:
param_dist = {
    "n_estimators": [200, 400, 600, 800],
    "max_depth": [3, 4, 5, 6, 8],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "subsample": [0.7, 0.85, 1.0],
    "colsample_bytree": [0.7, 0.85, 1.0],
    "min_child_weight": [1, 3, 5],
}

tuned_models = {}
for tgt in TARGETS:
    search = RandomizedSearchCV(
        XGBRegressor(random_state=RANDOM_STATE, verbosity=0),
        param_dist,
        n_iter=10,
        cv=5,
        scoring="r2",
        random_state=RANDOM_STATE,
        n_jobs=1
    )
    search.fit(X_train_tr, y_train[tgt])
    tuned_models[tgt] = search.best_estimator_
    print(f"Target: {tgt} | Best CV R²: {search.best_score_:.4f} | Best Params: {search.best_params_}")


### 4. Feature Importance Analysis


In [ ]:
feature_names = [c for c in features_cols if c not in ["Orientation", "Glazing_Area_Distribution"]] + [
    "Wall_Surface_Ratio", "Glazing_x_Height", "Orientation", "Glazing_Area_Distribution"
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, tgt in enumerate(TARGETS):
    model = tuned_models[tgt]
    importances = pd.Series(model.feature_importances_).sort_values(ascending=True)
    importances.plot(kind="barh", ax=axes[i], color="cornflowerblue")
    axes[i].set_title(f"XGBoost Feature Importance: {tgt}")

plt.tight_layout()
plt.show()


### 5. Production Model Bundle Creation


In [ ]:
bundle = joblib.load("artifacts/production/model_bundle.joblib")
print("Model Bundle loaded successfully.")
print("Bundle Keys:", list(bundle.keys()))
print("Algorithms available in Heating:", list(bundle['models']['heating'].keys()))
print("Algorithms available in Cooling:", list(bundle['models']['cooling'].keys()))
